# Rad-Scribe Pro — Notebook 1: Data Understanding & EDA
**Symbiosis Institute of Technology | Dept. of AI & ML**

Team: Tejas Kale · Hardik Gulati · Swaraj Deogirkar | Mentor: Dr. Zulfikar Ali Ansari

### Why this notebook exists
Before building any model we need to deeply understand what we're working with.
This notebook answers:
- What does the dataset look like?
- How long are the reports?
- Is there class imbalance?
- Are images consistent?
- What medical terms appear most?

These observations directly justify every architectural decision in Models A, B, C, D.

> Dataset: Indiana University Chest X-Ray Collection (IU X-Ray)
> Source: HuggingFace — MLforHealthcare/Indiana_University_Chest_X-ray_Collection

In [ ]:
!pip install -q datasets pandas matplotlib seaborn wordcloud transformers Pillow
print('done')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from datasets import load_dataset
from transformers import GPT2Tokenizer
from PIL import Image
import re

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size']  = 11

print('Loading IU X-Ray dataset from HuggingFace...')
raw = load_dataset('MLforHealthcare/Indiana_University_Chest_X-ray_Collection')
print(raw)
print(f'\nSplits: {list(raw.keys())}')
print(f'Train samples: {len(raw["train"]):,}')
print(f'Test  samples: {len(raw["test"]):,}')

---
## Section 1 — Dataset Structure

In [ ]:
# inspect first sample
sample = raw['train'][0]
print('Keys in each sample:')
for k, v in sample.items():
    if k == 'image':
        print(f'  image  : PIL Image {v.size} mode={v.mode}')
    else:
        print(f'  {k:10s}: {str(v)[:120]}')

In [ ]:
# build flat dataframe from all splits
rows = []
for split in raw.keys():
    for idx in range(len(raw[split])):
        item   = raw[split][idx]
        report = str(item.get('report', '') or '')
        rows.append({
            'hf_index' : idx,
            'split'    : split,
            'report'   : report,
            'img_width': item['image'].size[0],
            'img_height': item['image'].size[1],
            'img_mode' : item['image'].mode,
        })

df = pd.DataFrame(rows)
print(f'Total samples : {len(df):,}')
print(f'\nSplit distribution:')
print(df['split'].value_counts())
print(f'\nSample entries:')
df.head(5)

---
## Section 2 — Visualize Sample X-Rays with Reports

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

indices = [0, 5, 10, 20, 50, 100, 200, 300]
for i, idx in enumerate(indices):
    item   = raw['train'][idx]
    img    = item['image'].convert('RGB')
    report = str(item.get('report', ''))[:120]

    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'Sample {idx}', fontsize=9, fontweight='bold')
    axes[i].set_xlabel(report + '...', fontsize=7, wrap=True)
    axes[i].axis('off')

plt.suptitle('IU X-Ray — Sample Images with Report Snippets',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# print 5 full reports to understand structure
print('='*70)
print('SAMPLE RADIOLOGY REPORTS (full text)')
print('='*70)
for i in [0, 10, 50, 100, 200]:
    item = raw['train'][i]
    print(f'\n[Sample {i}]')
    print(item.get('report', 'NO REPORT'))
    print('-'*50)

---
## Section 3 — Report Text Analysis

In [ ]:
# word count per report
df['word_count'] = df['report'].apply(lambda x: len(str(x).split()))
df['char_count'] = df['report'].apply(lambda x: len(str(x)))

# token count using GPT-2 tokenizer (what our model sees)
tokenizer = GPT2Tokenizer.from_pretrained('distilgpt2')
tokenizer.pad_token = tokenizer.eos_token
df['token_count'] = df['report'].apply(
    lambda x: len(tokenizer(str(x))['input_ids']))

print('Report length statistics:')
print(df[['word_count','char_count','token_count']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# word count
axes[0].hist(df['word_count'], bins=40, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].axvline(df['word_count'].median(), color='red', linestyle='--',
                linewidth=2, label=f'Median={df["word_count"].median():.0f}')
axes[0].set_xlabel('Word Count'); axes[0].set_ylabel('Frequency')
axes[0].set_title('Report Word Count Distribution', fontweight='bold')
axes[0].legend()

# token count
axes[1].hist(df['token_count'], bins=40, color='#4CAF50', edgecolor='white', alpha=0.8)
axes[1].axvline(128, color='red', linestyle='--', linewidth=2, label='max_seq_len=128')
axes[1].axvline(df['token_count'].median(), color='orange', linestyle='--',
                linewidth=2, label=f'Median={df["token_count"].median():.0f}')
axes[1].set_xlabel('Token Count (GPT-2)'); axes[1].set_ylabel('Frequency')
axes[1].set_title('Token Count Distribution', fontweight='bold')
axes[1].legend()

# cumulative coverage
sorted_tokens = np.sort(df['token_count'].values)
cumulative    = np.arange(1, len(sorted_tokens)+1) / len(sorted_tokens)
axes[2].plot(sorted_tokens, cumulative * 100, color='#9C27B0', linewidth=2)
axes[2].axvline(128, color='red', linestyle='--', linewidth=2, label='128 tokens')
pct_covered = (df['token_count'] <= 128).mean() * 100
axes[2].axhline(pct_covered, color='orange', linestyle=':', linewidth=1.5,
                label=f'{pct_covered:.1f}% covered at 128')
axes[2].set_xlabel('Max Token Length'); axes[2].set_ylabel('% Reports Covered')
axes[2].set_title('Cumulative Coverage vs Token Length', fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Report Length Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'\nInsight: {pct_covered:.1f}% of reports fit within 128 tokens — justifies MAX_SEQ_LEN=128')

In [ ]:
# word frequency analysis
all_words = []
for report in df['report']:
    words = re.findall(r'\b[a-zA-Z]{3,}\b', str(report).lower())
    all_words.extend(words)

word_freq = Counter(all_words)

# remove stop words
STOP = {'the','and','are','was','with','this','that','from','have',
        'has','not','for','its','there','been','also','both','but',
        'nor','which','than','our','their','all','any','more','can'}
medical_freq = {w: c for w, c in word_freq.items() if w not in STOP}

top_30 = dict(sorted(medical_freq.items(), key=lambda x: x[1], reverse=True)[:30])

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(top_30.keys(), top_30.values(), color='#2196F3', edgecolor='white')
ax.set_xticklabels(top_30.keys(), rotation=45, ha='right')
ax.set_ylabel('Frequency')
ax.set_title('Top 30 Most Frequent Words in Radiology Reports', fontweight='bold')
# highlight medical terms in different color
MEDICAL = {'heart','lungs','normal','cardiac','pleural','effusion',
           'pneumothorax','consolidation','opacity','cardiomegaly',
           'mediastinum','airspace','silhouette','vasculature','atelectasis'}
for bar, word in zip(bars, top_30.keys()):
    if word in MEDICAL:
        bar.set_color('#F44336')
ax.legend(handles=[
    plt.Rectangle((0,0),1,1,color='#2196F3', label='Common words'),
    plt.Rectangle((0,0),1,1,color='#F44336', label='Medical terms')
])
plt.tight_layout()
plt.show()

In [ ]:
# medical term frequency specifically
PATHOLOGY_TERMS = [
    'normal','cardiac','heart','lungs','pleural','effusion','pneumothorax',
    'consolidation','opacity','cardiomegaly','mediastinum','atelectasis',
    'pneumonia','edema','nodule','fracture','infiltrate','lesion','enlarged',
    'silhouette','vasculature','costophrenic','airspace','bilateral'
]

med_counts = {term: word_freq.get(term, 0) for term in PATHOLOGY_TERMS}
med_counts = dict(sorted(med_counts.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(14, 5))
colors  = ['#F44336' if v > 1000 else '#FF9800' if v > 500 else '#4CAF50'
           for v in med_counts.values()]
ax.bar(med_counts.keys(), med_counts.values(), color=colors, edgecolor='white')
ax.set_xticklabels(med_counts.keys(), rotation=45, ha='right')
ax.set_ylabel('Frequency in all reports')
ax.set_title('Medical Term Frequency in IU X-Ray Reports', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 5 most common medical terms:')
for term, count in list(med_counts.items())[:5]:
    print(f'  {term:20s}: {count:,} occurrences')

---
## Section 4 — Class Imbalance Analysis

In [ ]:
PATHOLOGY_KW = [
    'pneumonia','effusion','cardiomegaly','opacity','infiltrate',
    'consolidation','atelectasis','pneumothorax','edema','fracture',
    'nodule','mass','lesion','enlarged','abnormal','disease','fibrosis'
]

def classify(r):
    t = str(r).lower()
    if any(k in t for k in PATHOLOGY_KW): return 'Abnormal'
    if any(w in t for w in ['normal','no significant','unremarkable',
                             'clear','no acute']): return 'Normal'
    return 'Unclear'

df['label'] = df['report'].apply(classify)

print('Class distribution (full dataset):')
print(df['label'].value_counts())
print()
print('Class distribution (train split):')
print(df[df['split']=='train']['label'].value_counts())
print()

# percentage
counts = df['label'].value_counts()
total  = len(df)
for label, count in counts.items():
    print(f'  {label:10s}: {count:,} ({count/total*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# pie chart
colors_pie = ['#4CAF50', '#F44336', '#FF9800']
counts_val = df['label'].value_counts()
axes[0].pie(counts_val.values, labels=counts_val.index,
            colors=colors_pie, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Class Distribution — Full Dataset', fontweight='bold')

# bar chart by split
split_label = df.groupby(['split','label']).size().unstack(fill_value=0)
split_label.plot(kind='bar', ax=axes[1], color=['#F44336','#4CAF50','#FF9800'],
                 edgecolor='white')
axes[1].set_xlabel('Split'); axes[1].set_ylabel('Count')
axes[1].set_title('Class Distribution by Split', fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Label')

plt.suptitle('Class Imbalance Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

normal_pct = (df['label'] == 'Normal').mean() * 100
print(f'\nCRITICAL INSIGHT: {normal_pct:.1f}% of reports are Normal')
print('This causes the LAZY MODEL PROBLEM — model learns to always predict Normal')
print('Solution: WeightedRandomSampler + weighted loss during training')

---
## Section 5 — Image Analysis

In [ ]:
# analyze image properties
print('Image resolution analysis...')
print(f'Width  — min:{df["img_width"].min()}  max:{df["img_width"].max()}  mean:{df["img_width"].mean():.0f}')
print(f'Height — min:{df["img_height"].min()}  max:{df["img_height"].max()}  mean:{df["img_height"].mean():.0f}')
print(f'Mode   — {df["img_mode"].value_counts().to_dict()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['img_width'],  bins=20, color='#2196F3', alpha=0.7, label='Width')
axes[0].hist(df['img_height'], bins=20, color='#F44336', alpha=0.7, label='Height')
axes[0].axvline(224, color='black', linestyle='--', linewidth=2, label='Resize target=224')
axes[0].set_xlabel('Pixels'); axes[0].set_ylabel('Count')
axes[0].set_title('Image Resolution Distribution', fontweight='bold')
axes[0].legend()

# pixel distribution on sample images
pixel_means = []
pixel_stds  = []
for i in range(0, min(200, len(raw['train'])), 5):
    img = np.array(raw['train'][i]['image'].convert('L'))
    pixel_means.append(img.mean())
    pixel_stds.append(img.std())

axes[1].scatter(pixel_means, pixel_stds, alpha=0.5, color='#9C27B0', s=20)
axes[1].set_xlabel('Mean Pixel Value (0-255)')
axes[1].set_ylabel('Std Pixel Value')
axes[1].set_title('Pixel Distribution (40 sample images)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nInsight: Images are grayscale but stored as RGB')
print('We normalize using ImageNet stats [0.485,0.456,0.406] / [0.229,0.224,0.225]')
print('All images resized to 224x224 for consistent model input')

In [ ]:
# visualize pixel histogram on a single image
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for col, idx in enumerate([0, 100, 500]):
    img = raw['train'][idx]['image'].convert('RGB')
    img_arr = np.array(img)

    axes[col].hist(img_arr.flatten(), bins=50,
                   color=['#2196F3','#4CAF50','#F44336'][col],
                   alpha=0.7, edgecolor='white')
    axes[col].set_title(f'Sample {idx} — Pixel Distribution', fontweight='bold')
    axes[col].set_xlabel('Pixel Value (0-255)')
    axes[col].set_ylabel('Count')
    axes[col].axvline(img_arr.mean(), color='black', linestyle='--',
                      linewidth=1.5, label=f'Mean={img_arr.mean():.0f}')
    axes[col].legend()

plt.suptitle('Pixel Intensity Distribution (3 samples)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 6 — Report Noise Analysis

In [ ]:
# check for XXXX tokens (anonymisation)
df['has_xxxx']   = df['report'].str.contains('xxxx', case=False, na=False)
df['has_empty']  = df['report'].str.strip().str.len() < 5
df['has_numbers'] = df['report'].str.contains(r'\d', na=False)

print('Report noise analysis:')
print(f'  Reports with XXXX tokens  : {df["has_xxxx"].sum():,} ({df["has_xxxx"].mean()*100:.1f}%)')
print(f'  Empty/missing reports     : {df["has_empty"].sum():,} ({df["has_empty"].mean()*100:.1f}%)')
print(f'  Reports with numbers      : {df["has_numbers"].sum():,} ({df["has_numbers"].mean()*100:.1f}%)')

# sample noisy reports
print('\nSample reports with XXXX tokens:')
noisy = df[df['has_xxxx']]['report'].head(3)
for i, r in enumerate(noisy):
    print(f'  [{i+1}] {r[:150]}')

print('\nInsight: XXXX tokens are hospital anonymisation placeholders')
print('These must be removed during preprocessing to avoid the model learning to generate XXXX')

---
## Section 7 — Key Insights Summary

In [ ]:
print('=' * 65)
print('RAD-SCRIBE PRO — DATA UNDERSTANDING INSIGHTS')
print('=' * 65)

normal_count   = (df['label'] == 'Normal').sum()
abnormal_count = (df['label'] == 'Abnormal').sum()
pct_covered    = (df['token_count'] <= 128).mean() * 100

insights = [
    ('Dataset size',
     f'{len(df):,} image-report pairs ({len(raw["train"]):,} train, {len(raw["test"]):,} test)'),
    ('Class imbalance',
     f'{normal_count/len(df)*100:.1f}% Normal vs {abnormal_count/len(df)*100:.1f}% Abnormal'
     ' — CAUSES LAZY MODEL PROBLEM'),
    ('Report length',
     f'Median {df["word_count"].median():.0f} words, {df["token_count"].median():.0f} tokens'
     f' — {pct_covered:.1f}% fit within 128 tokens'),
    ('Report noise',
     f'{df["has_xxxx"].sum()} reports contain XXXX anonymisation tokens — must be cleaned'),
    ('Missing reports',
     f'{df["has_empty"].sum()} empty/missing reports — must be dropped'),
    ('Image format',
     f'All grayscale stored as RGB, variable resolution — must resize to 224x224'),
    ('Dominant terms',
     'normal, heart, lungs, clear — biases model toward Normal predictions'),
    ('Vocabulary',
     f'{len(word_freq):,} unique words, highly specialised medical vocabulary'),
]

for title, detail in insights:
    print(f'\n  [{title}]')
    print(f'    {detail}')

print()
print('=' * 65)
print('ARCHITECTURAL DECISIONS JUSTIFIED BY THIS ANALYSIS')
print('=' * 65)
decisions = [
    'MAX_SEQ_LEN=128 covers 95%+ of reports without truncation',
    'WeightedRandomSampler needed to fix 70% Normal bias',
    'XXXX token removal required in preprocessing',
    'Image resize to 224x224 for consistent encoder input',
    'BERTScore used because medical paraphrasing is common',
    'RAG needed because domain vocabulary is narrow and repetitive',
]
for d in decisions:
    print(f'  ✓ {d}')
print('=' * 65)